In [1]:
import json
import random
from pathlib import Path


def create_brats2018_hgg_lgg_json(data_root, file_path, split_ratio):
    """cerate brats 2018 classifiy json"""
    data_root = Path(data_root)
    
    file = {'train':[], 'val':[]}
    hgg, lgg = [], []
    for ids, label in enumerate(data_root.glob("*")):
        print(label)
        if label.is_file():
            continue
       
        for patient in label.glob("*"):
            sample = {'images':[]}
            
            for modality in patient.glob("*t1.*"):
                sample['images'].append(str(modality))
            for modality in patient.glob("*t1ce.*"):
                sample['images'].append(str(modality))
            for modality in patient.glob("*t2.*"):
                sample['images'].append(str(modality))
            for modality in patient.glob("*flair.*"):
                sample['images'].append(str(modality))
            for modality in patient.glob("*seg.*"):
                sample['seg'] = str(modality)
            sample['label'] = ids
            if(len(sample['images']) != 4):
                print(sample)
            
            if ids==0:
                hgg.append(sample)
            else:
                lgg.append(sample)

    min_len = min(len(hgg), len(lgg))
    hgg = hgg[:min_len]
    num_train = int(split_ratio*min_len)
    file['train'] += hgg[: num_train] + lgg[: num_train]
    file['val'] += hgg[num_train: ] + lgg[num_train: ]
    random.shuffle(file['train'])
    random.shuffle(file['val'])
    file_path = Path(file_path)
    file_path.write_text(
        json.dumps(file, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )
    
create_brats2018_hgg_lgg_json('/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/MICCAI_BraTS_2018_Data_Training',
                              '/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/brats2018_hgg_lgg.json', 
                              0.7)

/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/MICCAI_BraTS_2018_Data_Training/LGG
/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/MICCAI_BraTS_2018_Data_Training/HGG


In [2]:
import json

def keep_only_t1ce(input_json_path, output_json_path):
    with open(input_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    for split in data:
        for sample in data[split]:
            if 'images' in sample and isinstance(sample['images'], list) and len(sample['images']) >= 2:
                sample['images'] = [sample['images'][1]]  # 保留第二个元素（t1ce）
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# 使用示例
keep_only_t1ce('/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/brats2018.json',
              '/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/brats2018_t1ce.json')